<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/06.Karpathy Char LM-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
# Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
import os

# 사용할 GitHub 저장소 이름과 URL 설정
repository_name = 'nlp2026'
repository_url = f'https://github.com/kikim6114/{repository_name}.git'

# 항상 루트 경로(/content/)로 이동 후 확인
%cd /content/

# 저장소가 이미 존재하면 클론을 건너뛰고, 없으면 클론
if not os.path.exists(repository_name):
    !git clone {repository_url}
    print(f"{repository_name} 클론 완료")
else:
    print(f"{repository_name} 폴더가 이미 존재합니다. 클론을 건너뜁니다.")

# 클론된 저장소 폴더로 이동
%cd {repository_name}

<span style="font-size:3em; line-height:36px"><strong>RNNs and LSTMs for Character-Level Language Models
</strong></span>
- 이 자료는 Andrej Karpathy의 블로그 글 The Unreasonable Effectiveness of Recurrent eural Networks를 PyTorch로 구현한 것입니다.
- 원래 Karpathy는 Python과 Numpy만을 사용해서 모든 것을 scratch로 구현하였습니다.
- 여기서는 블로그에서 설명한 기본 모델만 실습합니다.

### Reference
- [The Unreasonable Effectiveness of Recurrent Neural Networks (Karpathy, 2015)](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

In [ ]:
# IPython 셀 출력 설정: 셀 내 모든 표현식의 결과를 출력하도록 변경 (기본값은 마지막 표현식만 출력)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 노트북 출력 영역을 전체 화면 너비로 확장하는 CSS 적용
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# matplotlib 그래프를 노트북 셀 내부에 인라인으로 표시
%matplotlib inline 

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn               # 신경망 레이어 및 손실 함수
import torch.optim as optim         # 옵티마이저 (Adam, SGD 등)
from torch.utils.data import Dataset, DataLoader  # 커스텀 데이터셋 및 배치 로더
import urllib.request               # URL에서 데이터 파일 다운로드

# GPU가 사용 가능하면 cuda, 아니면 cpu로 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

### torch.Tensor 연습 

In [ ]:
# 0~23까지의 정수로 3D numpy 배열 생성: shape = (3, 2, 4)
x = np.array(range(24), dtype=np.int32)
x.shape = (3, 2, 4)

# PyTorch Tensor로 변환 후 마지막 차원의 크기를 확인
xt = torch.Tensor(x)
xt.size()[-1]  # 결과: 4

In [ ]:
# Python의 다중 반환값(튜플) 예시
# 함수가 단일 값과 튜플을 함께 반환하는 패턴 - RNN의 (output, hidden) 반환 구조와 동일
def jj(x):
    return x, (2*x, 3*x)  # 첫 번째: 스칼라, 두 번째: (2배, 3배) 튜플

In [ ]:
# 다중 반환값을 각 변수에 언패킹
y1, y2 = jj(1)   # y1=1, y2=(2, 3)
x2, x3 = y2      # 중첩 튜플도 다시 언패킹 가능
print(y1, x2, x3)  # 결과: 1 2 3

In [ ]:
# 3차원 numpy 배열 생성 연습
# shape (3, 2, 4): 3개의 블록, 각 블록에 2행 4열
x = np.array(range(24), dtype=np.int32)
x.shape = (3, 2, 4)
x

In [ ]:
# 첫 번째 블록 선택: shape (2, 4) -> 0번 인덱스의 2x4 행렬
x[0]

In [ ]:
# 모든 블록에서 0번째 행만 추출: shape (3, 4)
# RNN에서 배치 전체의 특정 시간 스텝을 슬라이싱하는 것과 유사한 패턴
x[:, 0, :]

In [ ]:
# 모든 블록의 1번째 행 슬라이싱 → shape 확인
x[:, 1, :].shape  # 결과: (3, 4)

In [ ]:
# 음수 인덱싱으로 마지막 행 선택 (-1 = 마지막 인덱스)
# x[:, -1, :] == x[:, 1, :] (행이 2개이므로 동일)
x[:, -1, :]

In [ ]:
# ravel(): 다차원 배열을 1차원으로 평탄화 (flatten과 유사, 메모리 효율적)
# generate_seq에서 확률 분포 p를 1차원으로 만들 때 사용
x.ravel()

In [ ]:
# reshape(-1): 마지막 차원을 자동 계산하여 형태 변환 → (1, 1, 24)
y = x.reshape(1, 1, -1)
# squeeze(): 크기가 1인 차원을 모두 제거 → (24,)
y.squeeze().shape

## Karpathy's character level language model
<img src="http://karpathy.github.io/assets/rnn/charseq.jpeg" width="400" height="400">

## Data Preprocessing

In [ ]:
# Karpathy의 char-rnn 프로젝트에서 사용한 tiny shakespeare 데이터셋 다운로드
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'shakespeare_data.txt'
urllib.request.urlretrieve(url, filename)  # URL에서 파일을 로컬에 저장

# 텍스트 파일을 문자열로 읽어오기
data_file = open(filename, 'r')
raw_data = data_file.read()
data_file.close()

# 데이터 앞부분 200자 출력하여 내용 확인
print(raw_data[:200])

- Vocabulary의 각 char를 index로 매핑
- Character를 one-hot encoding

In [ ]:
# 데이터셋 전체 글자 수 계산
data_length = len(raw_data)

# 텍스트에 등장하는 고유 문자들을 정렬하여 vocabulary(사전) 구성
vocab = sorted(list(set(raw_data)))
vocab_size = len(vocab)  # 65개의 고유 문자 (소문자, 대문자, 특수문자 등)

# 문자 <-> 인덱스 양방향 매핑 딕셔너리 생성
# char_to_index: 문자를 정수 인덱스로 변환 (모델 입력 준비용)
char_to_index = { char:index for (index,char) in enumerate(vocab) }
# index_to_char: 정수 인덱스를 다시 문자로 변환 (모델 출력 해석용)
index_to_char = { index:char for (index,char) in enumerate(vocab) }

print(f"데이터셋의 총 char 수 = {data_length}")
print()
print(f"Vocabulary = {vocab}")
print()
print(f"Vocabulary 크기 = {vocab_size}")
print()
print(f"char_to_index = {char_to_index}")
print()
print(f"index_to_char = {index_to_char}")

- 여기서는 데이터를 단순히 일정한 길이의 sub-sequence들로 잘라내어 사용하지만, 이 방법으로는 좋은 성능을 얻지 못할 수 있다. 

In [ ]:
def create_one_hot(ind, length):
    """인덱스를 one-hot 벡터로 변환
    
    예: ind=2, length=25 → [0, 0, 1, 0, ..., 0] (길이 25의 벡터)
    RNN의 문자 입력 표현 방식: 각 문자를 고유한 이진 벡터로 표현
    """
    vec = np.zeros(length)  # 모두 0으로 초기화된 벡터 생성
    vec[ind] = 1            # 해당 인덱스 위치만 1로 설정
    return vec

# 테스트: 인덱스 2, 길이 25인 one-hot 벡터
create_one_hot(2, 25)

In [ ]:
def chunk_data(raw_data, seq_len):
    """원시 텍스트 데이터를 동일한 길이의 청크(부분 시퀀스)들로 분할
    
    전체 텍스트를 seq_len 길이의 조각들로 나눔.
    마지막에 seq_len보다 짧은 나머지는 버림 (// 정수 나눗셈 사용).
    """
    chunks = []

    for i in range(len(raw_data) // seq_len):  # 나머지 버림으로 균일한 크기 보장
        start = i * seq_len
        end = start + seq_len
        chunk = raw_data[start:end]             # 슬라이싱으로 해당 구간 추출
        chunks.append(chunk)
        
    return chunks

# 테스트: 처음 104자를 25글자 단위로 4개의 청크로 분할
chunked_data = chunk_data(raw_data[:104], 25)
chunked_data

In [ ]:
def convert_dataset(dataset, char_to_index, vocab_size):
    """문자열 시퀀스 데이터셋을 인덱스 배열과 one-hot 인코딩 배열로 변환
    
    Args:
        dataset: 문자열 청크들의 리스트
        char_to_index: 문자 → 인덱스 매핑 딕셔너리
        vocab_size: 전체 어휘 크기 (one-hot 벡터의 차원)
    
    Returns:
        ind_dataset:     shape (N, L)        — 각 문자의 정수 인덱스 (타겟 레이블용)
        one_hot_dataset: shape (N, L, vocab_size) — 각 문자의 one-hot 벡터 (모델 입력용)
    """
    # 인덱스 배열: (청크 수, 시퀀스 길이)
    ind_dataset = np.zeros((len(dataset), len(dataset[0])), dtype=np.int32)
    # one-hot 배열: (청크 수, 시퀀스 길이, 어휘 크기)
    one_hot_dataset = np.zeros((len(dataset), len(dataset[0]), vocab_size), dtype=np.float32)

    for i, seq in enumerate(dataset):
        # 각 문자를 인덱스로 변환
        ind_seq = [char_to_index[c] for c in seq]
        # 각 인덱스를 one-hot 벡터로 변환
        one_hot_seq = [create_one_hot(ind, vocab_size) for ind in ind_seq]
        
        ind_dataset[i, :] = np.array(ind_seq, dtype=np.float32)
        one_hot_dataset[i, :, :] = np.asarray(one_hot_seq, dtype=np.float32)

    return ind_dataset, one_hot_dataset

# 변환 테스트 및 결과 shape 확인
ind_dataset, one_hot_dataset = convert_dataset(chunked_data, char_to_index, vocab_size)
print(ind_dataset.shape)       # (4, 25) - 4개 청크, 각 25글자
print(one_hot_dataset.shape)   # (4, 25, 65) - 각 글자가 65차원 one-hot 벡터

one_hot_dataset = batch_size(4) x seq_length(25) x vector_size(65)

In [ ]:
class ShakespeareDataset(Dataset):
    """PyTorch Dataset: 셰익스피어 문자 시퀀스 데이터를 (입력, 타겟) 쌍으로 제공
    
    핵심 아이디어 - 입출력 오프셋:
      입력:  [c0, c1, c2, ..., c(L-2)]  → one-hot (길이 L-1)
      타겟:  [c1, c2, c3, ..., c(L-1)]  → 인덱스 (길이 L-1)
      즉, 각 시간 스텝에서 '다음 문자'를 예측하도록 학습 데이터를 구성
    """
    def __init__(self, inds, one_hot):
        self.inds = inds          # 정수 인덱스 텐서: (N, L)
        self.one_hot = one_hot    # one-hot 텐서: (N, L, vocab_size)

    def __len__(self):
        # 데이터셋의 총 샘플 수 (청크 수)
        return self.one_hot.size(0)

    def __getitem__(self, idx):
        # 입력: 첫 번째 ~ 끝에서 두 번째 문자까지의 one-hot (:-1)
        input_onehot = self.one_hot[idx, :-1, :]   # shape: (L-1, vocab_size)
        # 타겟: 두 번째 ~ 마지막 문자까지의 인덱스 (1:)
        # 입력보다 한 칸 앞선 문자를 예측하도록 오프셋 적용
        target_ind = self.inds[idx, 1:]             # shape: (L-1,)

        return input_onehot, target_ind

In [ ]:
# 전체 데이터를 25글자 단위 청크로 분할
CHUNK_LEN = 25

data_chunks = chunk_data(raw_data, CHUNK_LEN)
# 청크들을 인덱스 배열과 one-hot 배열로 변환
train_ind, train_oh = convert_dataset(data_chunks, char_to_index, vocab_size)
train_ind.shape, train_oh.shape  # (44615, 25), (44615, 25, 65)

In [ ]:
# numpy 배열을 PyTorch Tensor로 변환하고 GPU로 전송
# train_ind: long 타입 (CrossEntropyLoss/NLLLoss의 타겟은 정수 인덱스여야 함)
train_ind_tt = torch.Tensor(train_ind).long().to(device)
# train_oh: float 타입 (신경망 입력은 실수형이어야 함)
train_oh_tt = torch.Tensor(train_oh).float().to(device)

# PyTorch Dataset 객체 생성 (DataLoader에서 배치 단위로 불러올 준비)
train_set = ShakespeareDataset(train_ind_tt, train_oh_tt)

## Recurrent Neural Networks

$$ 
\begin{align} h_t &= W_{ih} x_t + W_{hh} h_{t-1} + b_{ih} + b_{hh}\\
 a_t &= \text{tanh}(h_t) \\
 o_t &= \text{softmax}(W_{ho} a_t + b_{ho}) 
 \end{align} 
$$
 
 
## Implementation

In [ ]:
class MyRNNCell(nn.Module):
    """단일 RNN 셀 구현
    
    수식:
        combined = [x(t); h(t-1)]          # 입력과 이전 hidden state 연결
        h(t) = tanh(W_ih * combined + b)    # 새로운 hidden state 계산
        o(t) = log_softmax(W_ho * h(t))     # 출력 확률 분포 (NLLLoss용 log 확률)
    """
    def __init__(self, obs_dim, hidden_size, output_dim):
        super().__init__()
        self.hidden_size = hidden_size
        
        # 입력 x(t)와 은닉 h(t-1)를 합쳐서 새로운 hidden state로 변환
        # 입력 차원: obs_dim(65) + hidden_size(100) = 165
        self.i2h = nn.Linear(obs_dim + hidden_size, hidden_size)  # (165 → 100)
        # hidden state를 출력 어휘 크기의 로짓으로 변환
        self.h2o = nn.Linear(hidden_size, output_dim)              # (100 → 65)

        self.tanh = nn.Tanh()
        # LogSoftmax: log(softmax(x)) - NLLLoss와 함께 사용 (수치 안정성 향상)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, data, hidden):
        """RNN Cell의 단일 시간 스텝 계산
        
        Args:
            data:   현재 시간 스텝의 입력 x(t), shape: (batch, obs_dim)
            hidden: 이전 hidden state h(t-1), shape: (batch, hidden_size)
        Returns:
            output: 다음 문자의 log 확률, shape: (batch, output_dim)
            hidden: 현재 hidden state h(t), shape: (batch, hidden_size)
        """
        # x(t)와 h(t-1)를 dim=1 방향으로 연결: (64, 65+100) = (64, 165)
        combined = torch.cat((data, hidden), 1)

        # hidden state 업데이트
        hidden = self.i2h(combined)   # (64, 165) → (64, 100)
        hidden = self.tanh(hidden)    # 활성화 함수: 값 범위를 [-1, 1]로 압축

        # 출력 계산
        output = self.h2o(hidden)     # (64, 100) → (64, 65)
        output = self.softmax(output) # log 확률로 변환

        return output, hidden

In [ ]:
class MyRNN(nn.Module):
    """시퀀스 전체를 처리하는 RNN 모델
    
    MyRNNCell을 시간 스텝 수만큼 반복 적용하여 전체 시퀀스에 대한 출력을 생성.
    각 시간 스텝의 출력과 hidden state를 저장하여 반환.
    """
    def __init__(self, obs_dim, hidden_size, output_dim):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_dim = output_dim

        # 단일 RNN 셀을 재사용하여 모든 시간 스텝에 동일한 가중치 적용 (weight sharing)
        self.rnn_cell = MyRNNCell(obs_dim, hidden_size, output_dim)

    def forward(self, x):
        """전체 입력 시퀀스에 대해 RNN을 순차적으로 실행
        
        Args:
            x: 입력 텐서, shape = (B, L, D)
               B: batch size (배치 크기)
               L: sequence length (시퀀스 길이, ShakespeareDataset에서 24)
               D: vocab_size (one-hot 벡터 차원, 65)
        Returns:
            output_arr: 각 시간 스텝의 출력 log 확률, shape: (B, L, output_dim)
            hidden_arr: 각 시간 스텝의 hidden state, shape: (B, L, hidden_size)
        """
        batch_size, seq_len, n_feat = x.size()  # (64, 24, 65)
        # [Q] seq_length가 처음 지정한 25에서 24로 왜 변했을까?
        # => ShakespeareDataset.__getitem__에서 입력을 [:-1]로 잘라 길이가 24가 됨
        
        # 각 시간 스텝의 출력/hidden을 저장할 배열 초기화
        output_arr = torch.zeros((batch_size, seq_len, self.output_dim))   # (64, 24, 65)
        hidden_arr = torch.zeros((batch_size, seq_len, self.hidden_size))  # (64, 24, 100)
        
        # 모델 내부에서 생성한 Tensor도 입력과 동일한 device에 있어야 함 (GPU/CPU 불일치 방지)
        output_arr = output_arr.float().to(x.device)
        hidden_arr = hidden_arr.float().to(x.device)

        # 첫 번째 시간 스텝의 hidden state를 0으로 초기화
        hidden = self.init_hidden(batch_size, x.device)  # (64, 100)

        # 시퀀스의 각 시간 스텝을 순차적으로 처리 (BPTT: Backpropagation Through Time)
        for i in range(seq_len):
            # x[:, i, :]: 배치 전체의 i번째 시간 스텝 → shape: (64, 65)
            output, hidden = self.rnn_cell(x[:, i, :], hidden)
            # output: (64, 65) — 다음 문자의 log 확률
            # hidden: (64, 100) — 다음 스텝으로 전달될 hidden state
            
            output_arr[:, i, :] = output
            hidden_arr[:, i, :] = hidden

        return output_arr, hidden_arr

    def init_hidden(self, batch_size, device):
        """배치 전체의 초기 hidden state를 0으로 생성"""
        return torch.zeros(batch_size, self.hidden_size, device=device)

## Training

In [ ]:
def generate_seq(model, init_char_one_hot, length):
    """학습된 RNN 모델로 문자 시퀀스를 자동 생성 (autoregressive 방식)
    
    Autoregressive: 이전 출력이 다음 입력이 됨.
    단계:
      1. RNN에서 다음 문자의 확률 분포를 구함
      2. 확률 분포에서 다음 문자를 샘플링
      3. 샘플링된 문자를 다음 시간 스텝의 입력으로 사용
      4. length번 반복
    
    Args:
        model: 학습된 MyRNN 모델
        init_char_one_hot: 시작 문자의 one-hot 텐서, shape: (1, 1, vocab_size)
        length: 생성할 문자 수
    """
    curr_char = init_char_one_hot  # (1, 1, 65)
    # 시작 문자를 one-hot에서 정수 인덱스로, 다시 문자로 변환하여 출력 문자열 초기화
    output = index_to_char[torch.argmax(curr_char.squeeze()).item()]

    for i in range(length):
        # 현재 문자를 RNN에 통과시켜 다음 문자의 log 확률 획득
        out, _ = model(curr_char)  # curr_char: (1, 1, 65) → out: (1, 1, 65)

        # LogSoftmax 출력을 exp()로 되돌려 실제 확률 분포로 변환
        # out[:, -1, :]: 마지막(유일한) 시간 스텝의 출력 → shape: (1, 65)
        # .cpu(): numpy는 CPU 텐서만 처리 가능
        # .detach(): gradient 계산 그래프에서 분리 (추론 시에는 gradient 불필요)
        p = np.exp(out[:, -1, :].cpu().detach().numpy())  # shape: (1, 65)
        
        # 확률 분포 p를 기반으로 어휘 중 하나를 무작위 샘플링
        # ravel()로 (65,) 1D 배열로 평탄화한 뒤 확률 가중치로 사용
        out_ind = np.random.choice(range(vocab_size), p=p.ravel())
        
        out_char = index_to_char[out_ind]
        output += out_char
        
        # 예측된 문자를 one-hot으로 변환하여 다음 시간 스텝의 입력으로 사용
        curr_char = create_one_hot(out_ind, vocab_size)
        # RNN 입력 형태 (1, 1, vocab_size)로 변환 후 GPU로 전송
        curr_char = torch.Tensor(curr_char).float().to(device).view(1, 1, -1)

    return output

In [ ]:
def train_loop(model, optimizer, train_loader, n_epochs, test_char=None):
    """RNN 모델 학습 루프
    
    Args:
        model:        학습할 MyRNN 모델
        optimizer:    Adam 옵티마이저
        train_loader: 배치 데이터를 제공하는 DataLoader
        n_epochs:     전체 데이터를 반복 학습할 횟수
        test_char:    에포크마다 시퀀스 생성에 사용할 시작 문자 (None이면 생략)
    """
    for epoch in range(n_epochs):
        avg_loss = []
        # 배치 단위로 학습: input_seq (64, 24, 65), target_ind (64, 24)
        for input_seq, target_ind in train_loader:
            optimizer.zero_grad()  # 이전 배치의 gradient 초기화

            output, _ = model(input_seq)  # output: (64, 24, 65) — 각 시간 스텝의 log 확률

            # NLLLoss: LogSoftmax 출력과 정수 인덱스 타겟을 입력으로 받음
            # output.transpose(1, 2): (64, 24, 65) → (64, 65, 24) — NLLLoss의 입력 형식
            # target_ind: (64, 24) — 각 시간 스텝의 정답 문자 인덱스
            loss = nn.NLLLoss()(output.transpose(1, 2), target_ind)

            loss.backward()  # 역전파: gradient 계산
            
            # Gradient Clipping: gradient norm이 5를 초과하면 5로 제한
            # RNN의 exploding gradient 문제를 방지
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            
            optimizer.step()   # 가중치 업데이트
            avg_loss.append(loss.item())

        print('\n*** Epoch {} : Avg Train Loss {}'.format(epoch, np.mean(avg_loss)))

        # 에포크마다 현재 모델로 샘플 시퀀스를 생성하여 학습 진행 상황을 정성적으로 확인
        if test_char is not None:
            gen_seq = generate_seq(model, test_char, 100)
            print("Generated Sequence:\n {}".format(gen_seq))

In [ ]:
# ── 하이퍼파라미터 설정 ──────────────────────────────────────────────
HIDDEN_SIZE = 100    # RNN hidden state 차원 수
N_EPOCH = 10         # 전체 데이터 반복 학습 횟수
LR = 0.01            # 학습률 (Adam 옵티마이저)
BATCH_SIZE = 64      # 미니배치 크기
SAMP_CHAR = 'a'      # 시퀀스 생성 시 사용할 시작 문자

# DataLoader: 데이터셋에서 BATCH_SIZE 단위로 데이터를 꺼내어 모델에 공급
# shuffle=True: 에포크마다 배치 순서를 무작위로 섞어 학습 안정성 향상
# 입력 시퀀스 길이는 ShakespeareDataset에서 [:-1]로 잘려 25 → 24로 줄어든 상태
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)

# ── 모델, 옵티마이저 초기화 ──────────────────────────────────────────
# 입력: vocab_size(65) 차원의 one-hot, 출력: vocab_size(65) 차원의 log 확률
model = MyRNN(vocab_size, HIDDEN_SIZE, vocab_size).to(device)
optim = torch.optim.Adam(model.parameters(), lr=LR)

# ── 시퀀스 생성용 초기 문자 준비 ────────────────────────────────────
# 'a'의 one-hot 벡터를 (1, 1, 65) 형태의 텐서로 변환하여 GPU로 전송
test_char = create_one_hot(char_to_index[SAMP_CHAR], vocab_size)
test_char_tt = torch.Tensor(test_char).view(1, 1, -1).float().to(device)

# ── 학습 시작 ────────────────────────────────────────────────────────
train_loop(model, optim, train_loader, N_EPOCH, test_char_tt)

- 지금 모델로는 강의 슬라이드에서도 보았던 아래의 Karpathy의 원래 모델 출력은 기대할 수 없다.

>PANDARUS:
Alas, I think he shall be come approached and the day
When little srain would be attain'd into being never fed,
And who is but a chain and subjects of his death,
I should not sleep.

>Second Senator:
They are away this miseries, produced upon my soul,
Breaking and strongly should be buried, when I perish
The earth and thoughts of many states.

>DUKE VINCENTIO:
Well, your wit is in the care of side and that.

>Second Lord:
They would be ruled after this chamber, and
my fair nues begun out of the fact, to be conveyed,
Whose noble souls I'll have the heart of the wars.

>Clown:
Come, sir, I will make did behold your worship.

>VIOLA:
I'll drink it.

- LSTM이나 GRU 등을 이용하고 hidden layer를 여러 층 사용하는 방법을 사용하도록 구현해보자.

# 과제(Homework 3)
- 제출 마감: 2026.5.3 오후11:59:59
- 평가: 100점 기준에서 차감
### 과제 내용
- LSTM cell을 직접 작성한다. 즉, 앞서 실습한 **Implementation**처럼 PyTorch의 LSTM 함수를 이용하지 않고 scratch로 작성하여 수행한다.
- LSTM 층을 2개 사용한다.
- hidden_size = 100
- batch_size = 64
- Learning rate = 0.01
- N_EPOCH = 20
- sequence length = 30